# ESS Round 8 data curation: separate outputs with and without official ESS weights

This notebook preserves the collaborator-approved ESS Round 8 belief construction and sample selection while producing two clearly distinguished respondent-level datasets:

1. `ess8_cca_initial_beliefs_without_weights.csv`
2. `ess8_cca_initial_beliefs_with_weights.csv`

The two files contain exactly the same respondents, demographics, belief values, and missingness information. The second file additionally retains the four official ESS weight variables: `dweight`, `pspwght`, `pweight`, and `anweight`.

The weight variables are metadata for later analyses. They are **not** belief variables and must not be included as nodes in CCA or belief-network inference.

## Step 1 — Import packages and define the project paths

The notebook assumes this project structure:

```text
data_curation/
├── data/
│   ├── ESS8e02_3/
│   │   └── ESS8e02_3.csv
│   └── processed/
└── notebooks/
    └── 01_ess8_data_curation.ipynb
```

The path code works when the notebook is launched either from the project root or from the `notebooks` folder.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal

CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Could not locate the project data folder. Run this notebook from the "
        "project root or from its notebooks folder."
    )

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = DATA_DIR / "ESS8e02_3" / "ESS8e02_3.csv"

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"Raw ESS8 file not found: {RAW_DATA_PATH}")

print("Project root:", PROJECT_ROOT)
print("Raw input:", RAW_DATA_PATH)
print("Processed output folder:", PROCESSED_DIR)

Project root: /Users/karan/Desktop/SSM-MERC/polarization/data_curation
Raw input: /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/ESS8e02_3/ESS8e02_3.csv
Processed output folder: /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/processed


## Step 2 — Load the raw ESS Round 8 file and check its basic structure

No rows or variables are modified in this step. It verifies that the expected raw file contains 44,387 respondents, 23 countries, and the four official ESS weight columns.

In [2]:
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)

weight_columns = ["dweight", "pspwght", "pweight", "anweight"]
missing_weight_columns = [column for column in weight_columns if column not in df.columns]

if missing_weight_columns:
    raise KeyError(f"Missing expected ESS weight columns: {missing_weight_columns}")

print("Raw dataframe shape:", df.shape)
print("Number of countries:", df["cntry"].nunique())
print("Missing values in raw weight columns:")
print(df[weight_columns].isna().sum())

assert len(df) == 44387
assert df["cntry"].nunique() == 23
assert df[weight_columns].notna().all().all()

Raw dataframe shape: (44387, 535)
Number of countries: 23
Missing values in raw weight columns:
dweight     0
pspwght     0
pweight     0
anweight    0
dtype: int64


## Step 3 — Define reusable cleaning and rescaling functions

ESS uses special values for refusal, do not know, no answer, and other non-substantive responses. Each variable is therefore restricted to its valid substantive range before being rescaled to 0–1.

- `scale_keep` preserves the original direction.
- `scale_reverse` reverses the original direction.
- All final belief variables are oriented so that higher values indicate more right-wing, conservative, or anti-progressive responses.

In [3]:
def to_numeric_clean(series):
    """Convert a series to numeric; non-numeric values become NaN."""
    return pd.to_numeric(series, errors="coerce")


def valid_range(series, minimum, maximum):
    """Keep substantive values within the specified inclusive range."""
    numeric = to_numeric_clean(series)
    return numeric.where(numeric.between(minimum, maximum))


def scale_keep(series, minimum, maximum):
    """Rescale valid values to 0–1 while preserving direction."""
    numeric = valid_range(series, minimum, maximum)
    return (numeric - minimum) / (maximum - minimum)


def scale_reverse(series, minimum, maximum):
    """Reverse-code valid values and rescale them to 0–1."""
    numeric = valid_range(series, minimum, maximum)
    return (maximum - numeric) / (maximum - minimum)


country_labels = {
    "AT": "Austria",
    "BE": "Belgium",
    "CH": "Switzerland",
    "CZ": "Czechia",
    "DE": "Germany",
    "EE": "Estonia",
    "ES": "Spain",
    "FI": "Finland",
    "FR": "France",
    "GB": "United Kingdom",
    "HU": "Hungary",
    "IE": "Ireland",
    "IL": "Israel",
    "IS": "Iceland",
    "IT": "Italy",
    "LT": "Lithuania",
    "NL": "Netherlands",
    "NO": "Norway",
    "PL": "Poland",
    "PT": "Portugal",
    "RU": "Russian Federation",
    "SE": "Sweden",
    "SI": "Slovenia",
}

## Step 4 — Construct identifiers, demographics, voting metadata, and official ESS weights

The official weight columns are copied directly from the raw ESS file; they are not recalculated.

The variable `urbanization` reverses the raw `domicil` scale so that higher values indicate a more urban setting. `vote_simple` records only whether the respondent voted, did not vote, or was ineligible. It is not Van Noord et al.'s later seven-category party-vote classification.

In [4]:
metadata = pd.DataFrame(index=df.index)

metadata["ess_row_id"] = np.arange(1, len(df) + 1)
metadata["idno"] = df["idno"]
metadata["cntry"] = df["cntry"]
metadata["country_name"] = metadata["cntry"].map(country_labels)
metadata["ess_unique_id"] = (
    metadata["cntry"].astype(str) + "_" + metadata["idno"].astype(str)
)

metadata["agea"] = valid_range(df["agea"], 0, 120)
metadata["gndr"] = valid_range(df["gndr"], 1, 2)
metadata["eisced"] = valid_range(df["eisced"], 1, 7)

metadata["education_3cat"] = np.select(
    [
        metadata["eisced"].isin([1, 2]),
        metadata["eisced"].isin([3, 4, 5]),
        metadata["eisced"].isin([6, 7]),
    ],
    [1, 2, 3],
    default=np.nan,
)

metadata["hinctnta"] = valid_range(df["hinctnta"], 1, 10)
metadata["rlgblg"] = valid_range(df["rlgblg"], 1, 2)

# Raw domicil: 1 = big city and 5 = farm/countryside.
domicil_raw = valid_range(df["domicil"], 1, 5)
metadata["urbanization"] = 6 - domicil_raw

metadata["blgetmg"] = valid_range(df["blgetmg"], 1, 2)
metadata["vote_simple"] = valid_range(df["vote"], 1, 3)

# Preserve the official ESS values exactly as supplied.
for column in weight_columns:
    metadata[column] = to_numeric_clean(df[column])

assert metadata[weight_columns].notna().all().all()

print("Metadata shape:", metadata.shape)
print("Official weight columns retained:", weight_columns)

Metadata shape: (44387, 18)
Official weight columns retained: ['dweight', 'pspwght', 'pweight', 'anweight']


## Step 5 — Construct the 20 ESS Round 8 belief variables

Multi-item beliefs are computed as row-wise means only when all constituent items are available (`skipna=False`). This reproduces the accepted ESS8 curation and the descriptive statistics in Supplementary Table A2.

In [5]:
belief_map = {
    "left_right_identification": ["lrscale"],
    "gender_inequality": ["mnrgtjb"],
    "anti_lgbt": ["freehms", "hmsfmlsh", "hmsacld"],
    "euroscepticism": ["euftf"],
    "anti_immigration": ["imsmetn", "imdfetn", "impcntr"],
    "anti_egalitarianism": ["gincdif", "dfincac", "smdfslv"],
    "benefits_harm_economy": ["sbstrec", "sbbsntx"],
    "benefits_harm_society": ["sbprvpv", "sbeqsoc"],
    "welfare_chauvinism": ["imsclbn"],
    "anti_economic_interventionism": ["gvslvol", "gvslvue", "gvcldcr"],
    "anti_social_benefits_low_income": ["bnlwinc"],
    "anti_social_benefits_parents": ["wrkprbf"],
    "educational_spending": ["eduunmp"],
    "anti_basic_income": ["basinc"],
    "anti_climate_change_taxes": ["inctxff"],
    "anti_climate_change_renewables": ["sbsrnen"],
    "anti_climate_ban_appliances": ["banhhap"],
    "climate_skepticism": ["ccnthum"],
    "authoritarianism": ["impsafe", "ipfrule", "ipbhprp", "ipstrgv", "imptrad"],
    "anti_libertarianism": ["impdiff", "ipadvnt", "ipcrtiv", "impfree", "ipudrst"],
}

belief_columns = list(belief_map)
all_belief_items = sorted({item for items in belief_map.values() for item in items})
tmp_items = pd.DataFrame(index=df.index)

# Left–right identification and gender inequality
tmp_items["lrscale"] = scale_keep(df["lrscale"], 0, 10)
tmp_items["mnrgtjb"] = scale_reverse(df["mnrgtjb"], 1, 5)

# Anti-LGBT
tmp_items["freehms"] = scale_keep(df["freehms"], 1, 5)
tmp_items["hmsfmlsh"] = scale_reverse(df["hmsfmlsh"], 1, 5)
tmp_items["hmsacld"] = scale_keep(df["hmsacld"], 1, 5)

# Euroscepticism and anti-immigration
tmp_items["euftf"] = scale_reverse(df["euftf"], 0, 10)
for variable in ["imsmetn", "imdfetn", "impcntr"]:
    tmp_items[variable] = scale_keep(df[variable], 1, 4)

# Anti-egalitarianism
tmp_items["gincdif"] = scale_keep(df["gincdif"], 1, 5)
tmp_items["dfincac"] = scale_reverse(df["dfincac"], 1, 5)
tmp_items["smdfslv"] = scale_keep(df["smdfslv"], 1, 5)

# Perceived economic and social harm of benefits
tmp_items["sbstrec"] = scale_reverse(df["sbstrec"], 1, 5)
tmp_items["sbbsntx"] = scale_reverse(df["sbbsntx"], 1, 5)
tmp_items["sbprvpv"] = scale_keep(df["sbprvpv"], 1, 5)
tmp_items["sbeqsoc"] = scale_keep(df["sbeqsoc"], 1, 5)

# Welfare and government intervention
tmp_items["imsclbn"] = scale_keep(df["imsclbn"], 1, 5)
for variable in ["gvslvol", "gvslvue", "gvcldcr"]:
    tmp_items[variable] = scale_reverse(df[variable], 0, 10)

tmp_items["bnlwinc"] = scale_reverse(df["bnlwinc"], 1, 4)
tmp_items["wrkprbf"] = scale_reverse(df["wrkprbf"], 1, 4)
tmp_items["eduunmp"] = scale_keep(df["eduunmp"], 1, 4)
tmp_items["basinc"] = scale_reverse(df["basinc"], 1, 4)

# Climate policy and climate scepticism
tmp_items["inctxff"] = scale_keep(df["inctxff"], 1, 5)
tmp_items["sbsrnen"] = scale_keep(df["sbsrnen"], 1, 5)
tmp_items["banhhap"] = scale_keep(df["banhhap"], 1, 5)

ccnthum = to_numeric_clean(df["ccnthum"])
ccnthum = ccnthum.where(ccnthum.isin([1, 2, 3, 4, 5]))
tmp_items["ccnthum"] = (5 - ccnthum) / 4

# Human-value items
for variable in ["impsafe", "ipfrule", "ipbhprp", "ipstrgv", "imptrad"]:
    tmp_items[variable] = scale_reverse(df[variable], 1, 6)

for variable in ["impdiff", "ipadvnt", "ipcrtiv", "impfree", "ipudrst"]:
    tmp_items[variable] = scale_keep(df[variable], 1, 6)

assert tmp_items[all_belief_items].min(skipna=True).min() >= 0
assert tmp_items[all_belief_items].max(skipna=True).max() <= 1

beliefs = metadata.copy()
for belief_name, item_list in belief_map.items():
    beliefs[belief_name] = tmp_items[item_list].mean(axis=1, skipna=False)

beliefs["n_belief_missing"] = beliefs[belief_columns].isna().sum(axis=1)
beliefs["n_belief_available"] = len(belief_columns) - beliefs["n_belief_missing"]

print("Constructed belief dataframe shape:", beliefs.shape)
print("Number of belief variables:", len(belief_columns))

Constructed belief dataframe shape: (44387, 40)
Number of belief variables: 20


## Step 6 — Apply the CCA preprocessing sample rule

Respondents are retained when:

1. age is at least 18, or age is missing; and
2. no more than two of the 20 constructed beliefs are missing.

This is the modified CCA preprocessing rule described by Van Noord et al.

In [6]:
adult_or_missing_age = beliefs["agea"].ge(18) | beliefs["agea"].isna()

cca_initial = beliefs.loc[
    adult_or_missing_age & (beliefs["n_belief_missing"] <= 2)
].copy()

assert len(cca_initial) == 37118
assert cca_initial["cntry"].nunique() == 23
assert len(belief_columns) == 20

print("Raw ESS N:", len(df))
print("Final CCA-ready N:", len(cca_initial))
print("Countries:", cca_initial["cntry"].nunique())
print("Belief variables:", len(belief_columns))

Raw ESS N: 44387
Final CCA-ready N: 37118
Countries: 23
Belief variables: 20


## Step 7 — Validate the 20 beliefs against Supplementary Table A2

The reproduced number of valid observations must match exactly. Means and standard deviations must match the published values when rounded to two decimal places.

In [7]:
reproduced_stats = (
    cca_initial[belief_columns]
    .agg(["count", "mean", "std"])
    .T
    .reset_index()
    .rename(
        columns={
            "index": "belief_variable",
            "count": "N_reproduced",
            "mean": "mean_reproduced",
            "std": "sd_reproduced",
        }
    )
)
reproduced_stats["N_reproduced"] = reproduced_stats["N_reproduced"].astype(int)

paper_stats = pd.DataFrame(
    [
        ["left_right_identification", 34248, 0.51, 0.22],
        ["gender_inequality", 37038, 0.23, 0.27],
        ["anti_lgbt", 36098, 0.34, 0.27],
        ["euroscepticism", 35848, 0.51, 0.27],
        ["anti_immigration", 36459, 0.45, 0.27],
        ["anti_egalitarianism", 36772, 0.38, 0.19],
        ["benefits_harm_economy", 35956, 0.49, 0.23],
        ["benefits_harm_society", 36801, 0.41, 0.22],
        ["welfare_chauvinism", 36441, 0.54, 0.26],
        ["anti_economic_interventionism", 36927, 0.24, 0.16],
        ["anti_social_benefits_low_income", 36399, 0.55, 0.27],
        ["anti_social_benefits_parents", 36026, 0.47, 0.24],
        ["educational_spending", 36227, 0.57, 0.25],
        ["anti_basic_income", 35785, 0.50, 0.27],
        ["anti_climate_change_taxes", 36832, 0.55, 0.31],
        ["anti_climate_change_renewables", 37028, 0.26, 0.26],
        ["anti_climate_ban_appliances", 36962, 0.36, 0.29],
        ["climate_skepticism", 36035, 0.39, 0.20],
        ["authoritarianism", 36631, 0.66, 0.17],
        ["anti_libertarianism", 36769, 0.35, 0.16],
    ],
    columns=["belief_variable", "N_paper", "mean_paper", "sd_paper"],
)

validation = paper_stats.merge(reproduced_stats, on="belief_variable", how="left")
validation["mean_reproduced_round2"] = validation["mean_reproduced"].round(2)
validation["sd_reproduced_round2"] = validation["sd_reproduced"].round(2)
validation["N_matches"] = validation["N_reproduced"] == validation["N_paper"]
validation["mean_matches_round2"] = (
    validation["mean_reproduced_round2"] == validation["mean_paper"]
)
validation["sd_matches_round2"] = (
    validation["sd_reproduced_round2"] == validation["sd_paper"]
)

assert validation["N_matches"].all()
assert validation["mean_matches_round2"].all()
assert validation["sd_matches_round2"].all()

print("All 20 belief variables match Supplementary Table A2.")
validation

All 20 belief variables match Supplementary Table A2.


,belief_variable,N_paper,mean_paper,sd_paper,N_reproduced,mean_reproduced,sd_reproduced,mean_reproduced_round2,sd_reproduced_round2,N_matches,mean_matches_round2,sd_matches_round2
0,left_right_identification,34248,0.51,0.22,34248,0.512920,0.223370,0.51,0.22,True,True,True
1,gender_inequality,37038,0.23,0.27,37038,0.227955,0.269111,0.23,0.27,True,True,True
2,anti_lgbt,36098,0.34,0.27,36098,0.336637,0.270381,0.34,0.27,True,True,True
3,euroscepticism,35848,0.51,0.27,35848,0.508968,0.266828,0.51,0.27,True,True,True
4,anti_immigration,36459,0.45,0.27,36459,0.451734,0.265234,0.45,0.27,True,True,True
5,anti_egalitarianism,36772,0.38,0.19,36772,0.380239,0.193550,0.38,0.19,True,True,True
6,benefits_harm_economy,35956,0.49,0.23,35956,0.490534,0.226702,0.49,0.23,True,True,True
7,benefits_harm_society,36801,0.41,0.22,36801,0.410132,0.218079,0.41,0.22,True,True,True
8,welfare_chauvinism,36441,0.54,0.26,36441,0.543090,0.258760,0.54,0.26,True,True,True
9,anti_economic_interventionism,36927,0.24,0.16,36927,0.244343,0.159874,0.24,0.16,True,True,True


## Step 8 — Create separate with-weights and without-weights datasets

The without-weights dataset retains the exact column structure of the previously accepted ESS8 output. The with-weights dataset adds the four official ESS columns immediately before the belief variables.

A strict consistency check confirms that removing the four weights from the with-weights dataset reproduces the without-weights dataset exactly.

In [8]:
cca_without_weights = cca_initial.drop(columns=weight_columns).copy()

base_columns = list(cca_without_weights.columns)
weight_insertion_position = base_columns.index("vote_simple") + 1

with_weights_column_order = (
    base_columns[:weight_insertion_position]
    + weight_columns
    + base_columns[weight_insertion_position:]
)
cca_with_weights = cca_initial[with_weights_column_order].copy()

assert_frame_equal(
    cca_with_weights.drop(columns=weight_columns).reset_index(drop=True),
    cca_without_weights.reset_index(drop=True),
    check_dtype=False,
    check_exact=True,
)
assert cca_with_weights[weight_columns].notna().all().all()

# Optional comparison with the previous accepted ESS8 output.
legacy_candidates = [
    DATA_DIR / "ess8_cca_initial_beliefs.csv",
    PROJECT_ROOT / "ess8_cca_initial_beliefs.csv",
]
legacy_path = next((path for path in legacy_candidates if path.exists()), None)

if legacy_path is not None:
    legacy_accepted = pd.read_csv(legacy_path, low_memory=False)
    assert_frame_equal(
        cca_without_weights.reset_index(drop=True),
        legacy_accepted.reset_index(drop=True),
        check_dtype=False,
        check_exact=False,
        rtol=0,
        atol=1e-15,
    )
    print("Legacy consistency check passed:", legacy_path)
else:
    print("Legacy output was not found; the optional legacy comparison was skipped.")

print("Without-weights shape:", cca_without_weights.shape)
print("With-weights shape:", cca_with_weights.shape)

Legacy output was not found; the optional legacy comparison was skipped.
Without-weights shape: (37118, 36)
With-weights shape: (37118, 40)


## Step 9 — Save the datasets, validation table, and documentation

The notebook does not overwrite the original accepted `ess8_cca_initial_beliefs.csv`. New files are written to `data/processed/`.

In [9]:
without_weights_path = (
    PROCESSED_DIR / "ess8_cca_initial_beliefs_without_weights.csv"
)
with_weights_path = (
    PROCESSED_DIR / "ess8_cca_initial_beliefs_with_weights.csv"
)
validation_path = PROCESSED_DIR / "ess8_belief_validation_against_paper.csv"
readme_path = PROCESSED_DIR / "README_ess8_curation.txt"

cca_without_weights.to_csv(without_weights_path, index=False)
cca_with_weights.to_csv(with_weights_path, index=False)
validation.to_csv(validation_path, index=False)

readme_text = f"""README: ESS8 CCA-ready belief datasets
=======================================

Purpose
-------
This curation reproduces the collaborator-approved ESS Round 8 preprocessing
for the CCA/network stage and provides separate files with and without the
official ESS survey-weight columns.

Raw input
---------
File: ESS8e02_3.csv
Raw respondents: {len(df)}
Raw countries: {df['cntry'].nunique()}

Final sample
------------
CCA-ready respondents: {len(cca_without_weights)}
Countries: {cca_without_weights['cntry'].nunique()}
Belief variables: {len(belief_columns)}

Sample rule
-----------
Respondents are retained if age is at least 18 or missing and no more than
two of the 20 constructed belief variables are missing.

Files
-----
1. ess8_cca_initial_beliefs_without_weights.csv
   Contains identifiers, country information, demographics, simple voting
   metadata, 20 belief variables, and belief-missingness counts. It contains
   no survey-weight columns.

2. ess8_cca_initial_beliefs_with_weights.csv
   Contains the same respondents and values as the without-weights file, plus
   dweight, pspwght, pweight, and anweight copied directly from the raw ESS
   file.

3. ess8_belief_validation_against_paper.csv
   Compares reproduced N, mean, and SD with Supplementary Table A2.

Official ESS weights
--------------------
dweight: design weight.
pspwght: post-stratification weight including the design weight.
pweight: population-size weight for cross-national aggregation.
anweight: ESS analysis weight supplied in the raw file.

The official weights are not recalculated. They are not belief variables and
must not be used as nodes in CCA or belief-network inference. The belief values
are not multiplied by the weights, and rows are not replicated.

Validation
----------
The final sample contains 37,118 respondents from 23 countries. All 20
belief-variable Ns match Supplementary Table A2 exactly; means and standard
deviations match when rounded to two decimal places. Removing the four weight
columns from the with-weights file yields the without-weights file exactly.
"""

readme_path.write_text(readme_text, encoding="utf-8")

print("Saved:")
print("-", without_weights_path)
print("-", with_weights_path)
print("-", validation_path)
print("-", readme_path)

Saved:
- /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/processed/ess8_cca_initial_beliefs_without_weights.csv
- /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/processed/ess8_cca_initial_beliefs_with_weights.csv
- /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/processed/ess8_belief_validation_against_paper.csv
- /Users/karan/Desktop/SSM-MERC/polarization/data_curation/data/processed/README_ess8_curation.txt


## Step 10 — Final verification summary

This final cell reads the two saved CSV files back from disk and confirms their dimensions, row identity, column identity, and weight completeness.

In [10]:
saved_without = pd.read_csv(without_weights_path, low_memory=False)
saved_with = pd.read_csv(with_weights_path, low_memory=False)

assert saved_without.shape == (37118, 36)
assert saved_with.shape == (37118, 40)
assert saved_with[weight_columns].notna().all().all()

assert_frame_equal(
    saved_with.drop(columns=weight_columns),
    saved_without,
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=1e-15,
)

print("ESS8 curation completed successfully.")
print("Without weights:", saved_without.shape)
print("With weights:", saved_with.shape)
print("All official weights complete:", saved_with[weight_columns].notna().all().all())

ESS8 curation completed successfully.
Without weights: (37118, 36)
With weights: (37118, 40)
All official weights complete: True
